[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ashakram05/ayeshaAkram-flyrank/blob/main/work/notebooks/w07_action_playbook.ipynb)

# ML-10 ? Content Action Playbook

This notebook continues directly from the validated FlyRank Lane 2 work in Weeks 1?6. It uses the repo?s existing queue and model outputs as the source of truth and turns them into a clearer action playbook for a reviewer.

The goal is not to reinvent the scoring logic. The goal is to make the ranking useful, honest, and safe: which content deserves attention first, why, and what a human must verify before acting.

> The notebook respects the earlier decisions: client-aware validation, the baseline rule, the model choice, and the evidence-based claim that the queue is a reviewer aid rather than a causal answer.


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

This section reads the final ranked queue generated by the repo?s Week-4?6 workflow and turns it into a reviewer-first summary. The key question is not ?what score did the model give?? but ?which content deserves attention first, and for what reason??


In [1]:
from pathlib import Path

import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "outputs").exists() or not (ROOT / "data").exists():
    for candidate in [ROOT.parent, ROOT.parent.parent]:
        if (candidate / "outputs").exists() and (candidate / "data").exists():
            ROOT = candidate
            break

QUEUE_PATH = ROOT / "outputs" / "refresh_queue_sample.csv"
MODEL_REPORT_PATH = ROOT / "outputs" / "model_report.md"

queue = pd.read_csv(QUEUE_PATH)
queue["final_reason_codes"] = queue["final_reason_codes"].fillna("general_refresh_review")
queue["reason_count"] = queue["final_reason_codes"].str.split("|").str.len()

print(f"Queue rows loaded: {len(queue):,}")
print(f"Actions present: {sorted(queue['suggested_action'].dropna().unique().tolist())}")
print(f"Confidence values: {sorted(queue['confidence'].dropna().unique().tolist())}")

ranked = queue.sort_values("final_refresh_score", ascending=False).head(12).copy()
ranked["final_reason_codes"] = ranked["final_reason_codes"].str.replace("|", ", ", regex=False)
print()
print(ranked[[
    "final_rank",
    "final_refresh_score",
    "best_model_probability",
    "suggested_action",
    "confidence",
    "final_reason_codes",
]].to_string(index=False))

action_counts = queue["suggested_action"].value_counts().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(9, 5))
action_counts.plot(kind="bar", ax=ax, color="#2F6F7C")
ax.set_title("Top review actions in the ranked queue")
ax.set_xlabel("Suggested action")
ax.set_ylabel("Count")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
out_dir = ROOT / "work" / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(out_dir / "action_mix.png", dpi=200)
print()
print(f"Saved action mix chart to {out_dir / 'action_mix.png'}")

Queue rows loaded: 200
Actions present: ['refresh', 'refresh_and_review_ctr', 'refresh_and_review_engagement']
Confidence values: ['high', 'medium']

 final_rank  final_refresh_score  best_model_probability       suggested_action confidence                                                                                                                                                         final_reason_codes
          1            81.636697                0.782079 refresh_and_review_ctr       high declining_with_demand, low_ctr_visible_page, low_engagement_visible_page, model_decline_risk, visible_model_opportunity, ctr_review_candidate, engagement_review_candidate
          2            81.447656                0.788105 refresh_and_review_ctr       high                                                           declining_with_demand, low_ctr_visible_page, model_decline_risk, visible_model_opportunity, ctr_review_candidate
          3            81.430346                0.847372 refresh

## 2. Intended use and limits

*Who uses this, for what ? and where it stops being valid.*

This queue is intended for one very specific purpose: a human reviewer triaging a backlog of content that may deserve a refresh. It is a decision-support ranking, not a command to publish, delete, or rewrite material.

The repository?s earlier work makes the bounds explicit:

- The target is `is_declining_label`, derived from `trend_direction == "down"` and therefore a proxy for content that is declining, not a guarantee of editorial failure.
- The final queue is built from observed signals only; it does not use titles, URLs, client names, or private business logic.
- The Week 6 validation audit confirms the ranking should be interpreted as directional evidence, not causal proof.
- The model is strongest as a prioritization aid for reviewer time, especially when paired with a human check on page quality, business intent, and content freshness.

The key limit is that a high score does not prove the page should be refreshed. It simply means the recorded signals suggest the page is a good candidate to inspect.


In [2]:
import re

REPORT_TEXT = MODEL_REPORT_PATH.read_text(encoding="utf-8")
print(REPORT_TEXT[:1200])

match = re.search(r"\| random_forest \| ([0-9.]+) \| ([0-9.]+) \| ([0-9.]+) \| ([0-9.]+) \| ([0-9.]+) \|", REPORT_TEXT)
if match:
    roc_auc, avg_precision, precision_at_50, recall, f1 = [float(x) for x in match.groups()]
    print()
    print("Observed best-model metrics from the repo report:")
    print(f"  ROC AUC: {roc_auc:.3f}")
    print(f"  Avg precision: {avg_precision:.3f}")
    print(f"  Precision@50: {precision_at_50:.3f}")
    print(f"  Recall: {recall:.3f}")
    print(f"  F1: {f1:.3f}")
else:
    print("Could not parse the best model metrics from the model report.")

confidence_counts = queue["confidence"].value_counts().reindex(["high", "medium", "low"], fill_value=0)
print()
print("Confidence distribution in the ranked queue:")
print(confidence_counts.to_string())

low_quality = queue.loc[queue["confidence"] == "low"].head(5)
print()
print("Low-confidence examples for manual review consideration:")
print(low_quality[["content_id", "confidence", "suggested_action", "best_model_probability", "final_refresh_score"]].to_string(index=False))


# FlyRank Refresh Opportunity Model Report

This report is generated from the bundled anonymized starter dataset (`data/raw/content_refresh_anonymized.csv`).
The model ranks existing content for refresh review. It does not use titles, URLs, client names, domains, or keywords.

## Data

- Rows scored: 30,000
- Declining-label rows: 16,262
- Declining-label rate: 0.542
- Split strategy used for validation: client_holdout
- Target: `is_declining_label`

## Model Comparison

Best model: `random_forest` selected by `precision_at_50`.

| Model | ROC AUC | Avg precision | Precision@50 | Recall | F1 |
|---|---:|---:|---:|---:|---:|
| decision_tree | 0.742 | 0.575 | 0.540 | 0.716 | 0.634 |
| logistic_regression | 0.700 | 0.522 | 0.400 | 0.567 | 0.566 |
| random_forest | 0.750 | 0.618 | 0.740 | 0.744 | 0.640 |
| baseline_rules | 0.627 | 0.468 | 0.240 | - | - |

## Final Queue

- High-confidence items: 3,605
- Medium-confidence items: 11,395
- Low-confidence items: 15,000
- `monitor` items: 13,09

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

This is the human-safe boundary. The queue is useful for prioritization, but the actual decision should still be reviewed by a person before anything is refreshed, expanded, or deprioritized.

The no-go checklist below reflects the real risks already visible in the repo and in the queue: low evidence, no position data, poor visibility, low engagement, or a page whose observed pattern is too weak for a confident action.


In [3]:
review_rules = {
    "no_position_data": (queue["avg_position"].isna()) | (queue["avg_position"] <= 0),
    "low_visibility": queue["impressions_90d"] < 500,
    "low_sessions": queue["sessions_90d"] < 10,
    "low_confidence": queue["confidence"].eq("low"),
}

manual_review = queue.loc[
    review_rules["no_position_data"]
    | review_rules["low_visibility"]
    | review_rules["low_sessions"]
    | review_rules["low_confidence"]
].copy()

manual_review = manual_review.sort_values(["confidence", "final_refresh_score"], ascending=[True, False])

print(f"Rows flagged for manual review: {len(manual_review):,}")
print()
print(manual_review[[
    "content_id",
    "client_id",
    "confidence",
    "suggested_action",
    "impressions_90d",
    "sessions_90d",
    "avg_position",
    "final_refresh_score",
]].head(10).to_string(index=False))

print()
print("Manual-review conditions observed in the queue:")
for name, mask in review_rules.items():
    print(f"  - {name}: {int(mask.sum()):,} items")

review_table = pd.DataFrame({
    "Check": [
        "Page still receives meaningful search impressions",
        "Sessions and engagement are strong enough to justify a refresh",
        "Average position is observed and interpretable",
        "Action matches the reason codes and not just the score",
        "Editorial intent and client context are still aligned with the recommendation"
    ],
    "Required_before_action": [
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes"
    ]
})
print()
print(review_table.to_string(index=False))


Rows flagged for manual review: 39

          content_id         client_id confidence       suggested_action  impressions_90d  sessions_90d  avg_position  final_refresh_score
content_d6570c51c9bd client_3fdba35f04     medium refresh_and_review_ctr             2498             9          10.1            81.430346
content_e04eb9549989 client_3fdba35f04     medium refresh_and_review_ctr             3393             5           3.6            80.873188
content_4d76cdb3387b client_3fdba35f04     medium refresh_and_review_ctr             1597             5           2.7            80.362748
content_b4f35d640b1c client_3fdba35f04     medium                refresh             3867             5          27.5            80.321757
content_ba6f9dfcbca1 client_3fdba35f04     medium                refresh             4366             8          20.1            80.282275
content_e76ac7fed711 client_3fdba35f04     medium refresh_and_review_ctr             1877             3           5.8            8

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

This queue is only as good as the signal it was built on. The repo?s earlier work already established the model selection criterion: the best model was chosen by `precision_at_50`, and the resulting report showed a strong lift over the baseline. The monitoring rules below simply translate that into a practical operational checklist.

If the data distribution shifts, the action mix changes sharply, or the top model loses its Precision@50 edge, the queue should be re-checked before broad use.


In [4]:
import re

REPORT_TEXT = MODEL_REPORT_PATH.read_text(encoding="utf-8")
report_metrics = re.findall(r"\| random_forest \| ([0-9.]+) \| ([0-9.]+) \| ([0-9.]+) \| ([0-9.]+) \| ([0-9.]+) \|", REPORT_TEXT)
if report_metrics:
    roc_auc, avg_precision, precision_at_50, recall, f1 = [float(x) for x in report_metrics[0]]
    print(f"Current model report metrics: ROC AUC={roc_auc:.3f}, Avg precision={avg_precision:.3f}, Precision@50={precision_at_50:.3f}")
else:
    print("Could not locate the random_forest metrics in the model report.")

confidence_counts = queue["confidence"].value_counts().reindex(["high", "medium", "low"], fill_value=0)
trigger_summary = pd.DataFrame({
    "Signal": [
        "High-confidence share",
        "Low-confidence share",
        "Queue drift check",
        "Precision@50 guardrail"
    ],
    "Observed_value": [
        float(confidence_counts.get("high", 0) / len(queue)),
        float(confidence_counts.get("low", 0) / len(queue)),
        "Compare with the last model report and current queue mix",
        "Re-check if Precision@50 falls materially below the current report"
    ],
    "Action": [
        "Review if the share swings sharply in either direction",
        "Inspect whether low-confidence rows are dominating the queue",
        "Check for score drift or a data shape change",
        "Trigger a refresh of the model and the playbook"
    ]
})
print()
print(trigger_summary.to_string(index=False))

queue["review_priority"] = queue["final_refresh_score"].rank(method="first", ascending=False).astype(int)
print()
print(queue[["final_rank", "review_priority", "confidence", "suggested_action", "final_refresh_score"]].head(10).to_string(index=False))


Current model report metrics: ROC AUC=0.750, Avg precision=0.618, Precision@50=0.740

                Signal                                                     Observed_value                                                       Action
 High-confidence share                                                              0.805       Review if the share swings sharply in either direction
  Low-confidence share                                                                0.0 Inspect whether low-confidence rows are dominating the queue
     Queue drift check           Compare with the last model report and current queue mix                 Check for score drift or a data shape change
Precision@50 guardrail Re-check if Precision@50 falls materially below the current report              Trigger a refresh of the model and the playbook

 final_rank  review_priority confidence       suggested_action  final_refresh_score
          1                1       high refresh_and_review_ctr            

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ ? your paper builds on these files.*

This section exports the queue and a compact summary to the working area so the model and its action playbook can be reused downstream. The exports are based on the files already generated by the repo?s Week-4?6 pipeline, not on a hard-coded copy of the final report.


In [5]:
from pathlib import Path
import json

WORK_OUTPUTS = ROOT / "work" / "outputs"
WORK_OUTPUTS.mkdir(parents=True, exist_ok=True)

queue_summary = {
    "n_rows": int(len(queue)),
    "action_counts": queue["suggested_action"].value_counts().sort_values(ascending=False).to_dict(),
    "confidence_counts": queue["confidence"].value_counts().reindex(["high", "medium", "low"], fill_value=0).to_dict(),
    "top_10_ranked_items": queue.sort_values("final_refresh_score", ascending=False).head(10)[
        ["final_rank", "content_id", "suggested_action", "confidence", "final_refresh_score", "best_model_probability", "final_reason_codes"]
    ].to_dict(orient="records"),
}

queue_out = queue.sort_values("final_refresh_score", ascending=False).head(200).copy()
queue_out.to_csv(WORK_OUTPUTS / "action_playbook_queue.csv", index=False)
(WORK_OUTPUTS / "action_playbook_summary.json").write_text(json.dumps(queue_summary, indent=2), encoding="utf-8")

print(f"Saved ranked queue export: {WORK_OUTPUTS / 'action_playbook_queue.csv'}")
print(f"Saved summary export: {WORK_OUTPUTS / 'action_playbook_summary.json'}")
print()
print(json.dumps(queue_summary["action_counts"], indent=2, sort_keys=True))


Saved ranked queue export: c:\Users\ashak\ayeshaAkram-flyrank\work\outputs\action_playbook_queue.csv
Saved summary export: c:\Users\ashak\ayeshaAkram-flyrank\work\outputs\action_playbook_summary.json

{
  "refresh": 35,
  "refresh_and_review_ctr": 130,
  "refresh_and_review_engagement": 35
}


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled ? markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime ? Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] This notebook reuses the repo?s actual FlyRank outputs and earlier methodology instead of inventing a new one
